# ILCOF-1 stacking and powder diffraction

Build ILCOF-1 from its prepared molecular fragments, compare two ways its layers can stack and inspect their powder X-ray diffraction (PXRD) patterns against experiment. Open this notebook in its own folder. Use a Python 3.12 kernel with the notebook extra installed from this checkout (`python -m pip install '.[notebook]'` from the repository root).

In [ ]:
import coflandscaper as cl

## Define the framework

ILCOF-1 forms a square (`sql`) net from a four-connected pyrene tetraaldehyde and p-phenylenediamine. The supplied XYZ files represent fragments of the linked framework. [ILCOF-1 synthesis](https://doi.org/10.1002/chem.201203753)

In [ ]:
COF_NAME = "ILCOF-1"
TOPOLOGY = "sql"

## Build one layer

The builder reads the prepared node from `0_node/ILCOF-1_node.xyz` and the linker from `0_linker/ILCOF-1_linker.xyz`. Helium atoms mark the sites where the fragments join and are removed during assembly. The node has four markers and the linker has two. When supplying your own fragments for this topology, place one He-marked XYZ directly in each folder.

The initial structure is saved in `ILCOF-1/1_ILCOF-1_single_layer/ILCOF-1_unopt.cif`.

In [ ]:
cl.BuildCOF2D().build(TOPOLOGY, COF_NAME)

## Relax the single layer

The default MACE-MH-1 calculator relaxes the in-plane cell lengths independently, holding the right angles and 15 Å cell height fixed. Its first use may download the model. A converged layer is saved as `ILCOF-1_preopt.cif` in stage 1. If convergence fails, this cell stops and reports the saved intermediate.

In [ ]:
cl.GeometryOpt().run_preopt(COF_NAME, topo=TOPOLOGY)

## Sample layer stacking

Scan layer separations from 3.0 to 4.0 Å in 0.1 Å steps. For this square net, the slip runs from eclipsed (AA) stacking to half the in-plane lattice diagonal (`(a+b)/2`). The grid uses 1 Å steps and includes the exact endpoint. If the endpoint falls less than 0.1 Å beyond a grid point, it replaces that point. Both stacking modes are saved under `ILCOF-1/2_ILCOF-1_matrix/`. Energy evaluation across this grid can take time. Advanced users can adjust the resolution in `CreateMatrix`.

In [ ]:
cl.CreateMatrix().run(COF_NAME, topo=TOPOLOGY)

## Calculate single-point energies

Evaluate each stacked geometry with the default MACE-MH-1 calculator. Each mode gets its own energy table in `ILCOF-1/3_ILCOF-1_landscape/`.

In [ ]:
cl.SinglePoint().run(COF_NAME)

## View the landscapes

The figures show how relative energy changes with separation and slip. Each star marks the lowest sampled energy for that mode. The figures are also saved in stage 3.

In [ ]:
cl.Landscape().run(COF_NAME, show=True)

## Select candidates

Choose the lowest sampled energy in each mode for relaxation. The selected CIFs go into the stage 3 `selection/` folder.

In [ ]:
cl.SelectCofs().run(COF_NAME)

## Relax the stacked candidates

Relax the atoms and the full unit cell of each selected structure. Stage 4 saves the CIFs and per-layer energies. If a calculation reaches the step limit, a warning identifies the intermediate geometry and the energy table records it in `stopped_due_to_max_steps`.

In [ ]:
cl.GeometryOpt().run(COF_NAME)

## Compare the final geometries

Inspect the interlayer distance (ILD, Å), lateral slip (ILS, Å) and energies of each candidate. The table is also saved as `ILCOF-1/5_ILCOF-1_analysis/final_structures.csv`.

In [ ]:
cl.AnalyzeStacking().analyze(COF_NAME)

## Compare diffraction patterns

The supplied experimental scan uses a Cu wavelength of 1.5406 Å. Simulate with the matching Cu Kα₁ preset. Stage 5 stores the simulated patterns under `pxrd_xy/`.

In [ ]:
cl.PXRD(wavelength="CuKa1").run(COF_NAME, export_hkl=True)

Compare the simulated reflections with the measured trace from `experimental_pxrd/`. Each candidate gets a comparison figure under `pxrd_plots/` in stage 5.

In [ ]:
cl.PXRD().plot_sim_vs_exp(COF_NAME)